In [ ]:

# Install & Import Libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:

# Dataset setup
# Download the dataset from Kaggle separately, then point WASTE_DATASET_PATH at its six-class directory.
# Expected subfolders: cardboard, glass, metal, paper, plastic, trash.
dataset_path = os.environ.get(
    "WASTE_DATASET_PATH",
    "data/raw/Garbage classification/Garbage classification",
)
if not os.path.isdir(dataset_path):
    raise FileNotFoundError(
        f"Dataset directory not found: {dataset_path}. "
        "Download the dataset and set WASTE_DATASET_PATH to its class-folder directory."
    )
print(os.listdir(dataset_path))


In [ ]:

# Load & Preprocess Images

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

image_size = 224
data, labels = [], []

class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
class_map = {name: idx for idx, name in enumerate(class_names)}

for class_name in class_names:
    class_path = os.path.join(dataset_path, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (image_size, image_size))
            img = preprocess_input(img.astype(np.float32))
            data.append(img)
            labels.append(class_map[class_name])

data = np.array(data)
labels = np.array(labels)

In [ ]:

#  Split Dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    data, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [ ]:

#  Data Augmentation
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
train_datagen.fit(X_train)

In [ ]:
# Configure a local model path (override with WASTE_MODEL_PATH if needed).
import os
model_path = os.environ.get('WASTE_MODEL_PATH', 'models/waste_model_v2.keras')


In [ ]:
# Train Model (Using Transfer Learning)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Input, Dense, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model_path = os.environ.get('WASTE_MODEL_PATH', 'models/waste_model_v2.keras')

if os.path.exists(model_path):
    print(" نموذج محفوظ موجود بالفعل، سيتم تحميله")
    model = load_model(model_path)
    print(" تم تحميل النموذج بنجاح، لا حاجة لإعادة التدريب")
else:
    print("(Transfer Learningباستخدام ) لا يوجد نموذج محفوظ، سيتم البدء في تدريب جديد")

    # 1. Load the previously trained base model (MobileNetV2)
    base_model = MobileNetV2(
        input_shape=(image_size, image_size, 3),
        include_top=False,  # We do not want to include the final ImageNet classifier layer
        weights='imagenet'
    )

    # 2. Freeze the base model
    base_model.trainable = False # We don't want to re train the layers that already learned the features

    # 3. Adding a classifier "head"
    # We implemented the Keras functional API
    inputs = Input(shape=(image_size, image_size, 3))
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(6, activation='softmax')(x) # 6 classes

    # 4. Create new model
    model = Model(inputs, outputs)

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # 5. Defining when to stop
    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=5,             # Stop after 5 epochs of no improvement
        restore_best_weights=True # Keep the best version of the model
    )
    # If the model gets "stuck", reduce the learning rate
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6
    )
    callbacks_list = [early_stop, reduce_lr]

    # 6. Train model
    history = model.fit(
        train_datagen.flow(X_train, y_train, batch_size=32),
        epochs=50,  # Increased epochs with the help of EarlyStopping to find the best one
        validation_data=(X_val, y_val),
        callbacks=callbacks_list # Use the callbacks from step 5
    )

    # 7. Save model locally to avoid retraining every time
    os.makedirs(os.path.dirname(model_path) or '.', exist_ok=True)
    model.save(model_path)
    print(" تم حفظ النموذج محلياً للاستخدام لاحقاً")


In [ ]:

# Evaluate Model

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f" دقة النموذج: {test_acc*100:.2f}%")


In [ ]:

#  Predict with Arabic Labels

arabic_labels = {
    'cardboard': 'كرتون',
    'glass': 'زجاج',
    'metal': 'معدن',
    'paper': 'ورق',
    'plastic': 'بلاستيك',
    'trash': 'نفايات'
}

def predict_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print(f" خطأ: تعذر تحميل الصورة من {img_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (image_size, image_size))
    img = preprocess_input(img.astype(np.float32))
    img = np.expand_dims(img, axis=0)
    prediction = model.predict(img)
    class_idx = np.argmax(prediction)
    class_name = class_names[class_idx]
    print(f" الفئة المتوقعة: {arabic_labels[class_name]} ({class_name})")

# Example usage (uncomment after confirming this file exists):
# predict_image(os.path.join(dataset_path, "glass", "glass1.jpg"))

In [ ]:
# Optional local demo. It is disabled by default; set RUN_DEMO=True intentionally.
RUN_DEMO = False

if RUN_DEMO:
    import gradio as gr

    def classify_image_text_only(image):
        import cv2, numpy as np
        img = cv2.resize(image, (image_size, image_size))
        img = preprocess_input(img.astype(np.float32))
        img = np.expand_dims(img, axis=0)
        prediction = model.predict(img)
        class_idx = np.argmax(prediction)
        class_name = class_names[class_idx]
        arabic_label = arabic_labels[class_name]
        return f" الفئة المتوقعة: {arabic_label}"

    interface_text = gr.Interface(
        fn=classify_image_text_only,
        inputs=gr.Image(type="numpy", label=" حمّل صورة النفايات"),
        outputs=gr.Textbox(label="النتيجة بالعربية"),
        title="نظام تصنيف النفايات بالصور",
        description="اختر صورة نفايات لمعرفة نوعها (ورق، بلاستيك، زجاج، كرتون، معدن، نفايات)."
    )
    interface_text.launch(share=False)
